
%md

####Objetivo del notebook

a. Realizar el tratamiento de datos faltantes si existen en las 15 variables categóricas, que por contraste gráfico de sus distribuciones entre el grupo no fatales y en el grupo muerte, se encuentran diferencias entre ellas.

Las 15 variables son:

1. Sexo de la victima - "sexo_victima" - sexo del lesionado dede la perspectiva biológica, se refiere a las características genéticas, endocrinas y morfológicas del cuerpo (Hombre, Mujer). 

2. Ciclo vital - "ciclo_vital" - etapa de desarrollo en que se encontraba la víctima (infancia, adolescencia, juventud, adultez, adulto mayor).

3. Escolaridad - "escolaridad" - grado de escolaridad más alto al cual ha llegado la persona de acuerdo con los niveles del sistema educativo formal: preescolar, básica en sus niveles de primaria, secundaria, media y superior.

4. Estado civil - "estado_civil" - Es la situación de cada persona en relación con las leyes o costumbres relativas al matrimonio que existen en el país (Separado (a), Divorciado (a), Viudo (a), Unión libre, Soltero (a), Casado (a))

5. Día del hecho - "dia_del_hecho" - Día de la semana en la que ocurrieron los hechos.

6. Departamento del hecho - "departamento_del_hecho_dane" - Departamento de Colombia donde ocurrió el hecho.
    
7. Zona del hecho - "zona_del_hecho" - Clasificación del territorio que diferencia los espacios comprendidos dentro del casco urbano de un municipio y los centros poblados (zona urbana), de los que están fuera de él (zona rural). Explícitamente se refiere a la zona donde se presentaron los hechos.

8. Escenario del hecho -"escenario_del_hecho"- Clasificación del lugar donde ocurrieron los hechos
    
9. Actividad durante el hecho - "actividad_durante_hecho" - Clasificación de las tareas u operaciones que se encontraba realizando la persona al momento de la lesión.
    
10. Contexto del hecho -"contexto_del_hecho" - Se refiere al contexto de violencia no fatal en la cual se produce la lesión o agresión (lesiones por violencia intrafamiliar, lesiones por violencia de pareja).
    
11. Mecanismo causal -"mecanismo_causal" - Corresponde a la clasificación de los tipos de mecanismos fisiopatológicos que conllevaron a la lesión de la persona. Se clasifican los casos de lesiones de acuerdo a la causa principal que la originó, pueden ser por algún tipo de trauma o lesión externa (contundente, mecanismo multiple, etc). 

12. Sexo del agresor - "sexo_del_agresor" - Se refiere a la variable biológica que clasifica a la población en hombres y mujeres.En este caso específico hace referencia al sexo del agresor, según el relato de la víctima.

13. Presunto agresor -"presunto_agresor" - Caracterización de la persona que se presume, o se sabe, ha sido el causante de la lesión. 

14. Factor desencadenante de la agresión - "factor_desencadenante_agresion" - Factor de riesgo y/o causa circunstancial, el cual desencadenó la agresión hacia la víctima, referido por la víctima (intolerancia, celos, etc). 

15. Incapacidad médico legal - "incapacidad_medicolegal" - Es el parámetro forense en Colombia basado en el tiempo en días que toma la reparación de las lesiones, en el marco de un proceso judicial por lesiones no fatales.

-Entrada:

Base ase de datos: vif_vp_no_fatales_v1, cuyo origen es el notebook 01_04.

-Salida:

b. Obtener y guardar la base de datos del grupo no fatales "victimas_VIF_VP_con_muerte_homicidio_v2" sin datos faltantes, con un total de 101 casos.



In [0]:
from pyspark.sql.functions import col, sum, count, when, create_map, lit, lower, trim
import matplotlib.pyplot as plt
from itertools import chain

In [0]:
#Leer tabla de datos como un data frame
 
df_victimas_VIF_VP_con_muerte_homicidio_v1= spark.table("ml_proyecto_7405607705157039.default.victimas_VIF_VP_con_muerte_homicidio_v1")

In [0]:
#Descripción de columnas

df_victimas_VIF_VP_con_muerte_homicidio_v1.printSchema()

In [0]:
df_victimas_VIF_VP_con_muerte_homicidio_v1.count()

In [0]:
#verificar valores unicos

columnas = [
    "sexo_victima",
    "ciclo_vital",
    "escolaridad",
    "estado_civil",
    "dia_del_hecho",
    "departamento_del_hecho_dane",
    "zona_del_hecho",
    "escenario_del_hecho",
    "actividad_durante_hecho",
    "contexto_del_hecho",
    "mecanismo_causal",
    "sexo_del_agresor",
    "presunto_agresor",
    "factor_desencadenante_agresion",
    "dias_de_incapacidad_medicolegal"

]

for c in columnas:
    print(f"\n===== {c} =====")

    n = df_victimas_VIF_VP_con_muerte_homicidio_v1.select(c).distinct().count()

    df_victimas_VIF_VP_con_muerte_homicidio_v1.select(c) \
        .distinct() \
        .orderBy(c) \
        .show(n=n, truncate=False)


In [0]:
# Volver el valor "sin información unico" en nuevas columnas

columnas = [
    "sexo_victima",
    "ciclo_vital",
    "escolaridad",
    "estado_civil",
    "dia_del_hecho",
    "departamento_del_hecho_dane",
    "zona_del_hecho",
    "escenario_del_hecho",
    "actividad_durante_hecho",
    "contexto_del_hecho",
    "mecanismo_causal",
    "sexo_del_agresor",
    "presunto_agresor",
    "factor_desencadenante_agresion",
    "dias_de_incapacidad_medicolegal"

]

for c in columnas:
    n = df_victimas_VIF_VP_con_muerte_homicidio_v1 = n = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
        f"{c}_cod",
        when(
            lower(trim(col(c))).isin("sin información"),
            "sin informacion"
        ).otherwise(col(c))
    )

In [0]:
df_victimas_VIF_VP_con_muerte_homicidio_v1.printSchema()

In [0]:
#Verificar que el valor sin informacion es unico en nuevas columnas

columnas = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"

]


for c in columnas:

    valores = (
        df_victimas_VIF_VP_con_muerte_homicidio_v1
        .select(c)
        .distinct()
        .orderBy(c)
    )

    n = valores.count()

    print(f"\n===== {c} ({n} valores distintos) =====")

    valores.show(n=n, truncate=False)


In [0]:
#conteo de valores sin información en columnas nuevas

columnas_cod = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"

]


conteo = df_victimas_VIF_VP_con_muerte_homicidio_v1.select([
    count(
        when(col(c) == "sin informacion", c)
    ).alias(c)
    for c in columnas_cod
])

display(conteo)

In [0]:
#filtrar en escolaridad_cod el valor "sin informacion"
df_escolar_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("escolaridad_cod") == "sin informacion")
display(df_escolar_sininfo)


In [0]:
#Cambiar el valor sin información de escolaridad por primaria
df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "escolaridad_cod",
    when(col("escolaridad_cod") == "sin informacion", "Primaria")
    .otherwise(col("escolaridad_cod"))
)

In [0]:
#filtrar en estado_civil_cod el valor "sin informacion"
df_ecivil_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("estado_civil_cod") == "sin informacion")
display(df_ecivil_sininfo)


In [0]:
#Cambiar el valor sin información de estado_civil_cod por Soltero (a)
df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "estado_civil_cod",
    when(col("estado_civil_cod") == "sin informacion", "Soltero (a)")
    .otherwise(col("estado_civil_cod"))
)

In [0]:
#filtrar en la columna "zona_del_hecho_cod" el valor "sin informacion"
df_zona_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("zona_del_hecho_cod") == "sin informacion")
display(df_zona_sininfo)


Los datos faltantes en zona_del_hecho_cod se van a completar en función a los casos predominantes segun departamento del hecho.

In [0]:
#filtrar df_victimas_VIF_VP_con_muerte_homicidio_v1 por el codigo del departamento que corresponde al Valle del Cauca
df_dpto_76valle = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("codigo_dane_departamento") == 76)

In [0]:
df_dpto_76valle.count()

In [0]:
# Gráfica de barras zona_del_hecho_cod en Valle del Cauca 

#Agrupar por categoría, contar casos y ordenar
df_zona_del_hecho_76valle = (
    df_dpto_76valle
    .groupBy("zona_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("zona_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_zona_del_hecho_76valle.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["zona_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos en el Valle del cauca segun la zona donde ocurrió el hecho")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#filtrar df_victimas_VIF_VP_con_muerte_homicidio_v1 por el codigo del departamento que corresponde al Casanare
df_dpto_85casanare = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("codigo_dane_departamento") == 85)

In [0]:
df_dpto_85casanare.count()

In [0]:
# Gráfica de barras zona_del_hecho_cod en Casanare

#Agrupar por categoría, contar casos y ordenar
df_zona_del_hecho_85casanare = (
    df_dpto_85casanare
    .groupBy("zona_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("zona_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_zona_del_hecho_85casanare.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["zona_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos en Casanare segun la zona donde ocurrió el hecho")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Se filtrar df_victimas_VIF_VP_con_muerte_homicidio_v1 por el codigo del departamento que corresponde al Magdalena
df_dpto_47magdalena = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("codigo_dane_departamento") == 47)

In [0]:
df_dpto_47magdalena.count()

In [0]:
# Gráfica de barras zona_del_hecho_cod en Magdalena 

#Agrupar por categoría, contar casos y ordenar
df_zona_del_hecho_47magdalena = (
    df_dpto_47magdalena
    .groupBy("zona_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("zona_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_zona_del_hecho_47magdalena.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["zona_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos en el Magdalena segun la zona donde ocurrió el hecho")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Se filtrar df_victimas_VIF_VP_con_muerte_homicidio_v1 por el codigo del departamento que corresponde La Guajira
df_dpto_44Guajira = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("codigo_dane_departamento") == 44)

In [0]:
df_dpto_44Guajira.count()

In [0]:
# Gráfica de barras zona_del_hecho_cod en La Guajira

#Agrupar por categoría, contar casos y ordenar
df_zona_del_hecho_44Guajira = (
    df_dpto_44Guajira
    .groupBy("zona_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("zona_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_zona_del_hecho_44Guajira.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["zona_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos en La Guajira segun la zona donde ocurrió el hecho")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Cambiar el valor sin información de zona_del_hecho_cod
df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "zona_del_hecho_cod",
    when(col("zona_del_hecho_cod") == "sin informacion", "Cabecera municipal")
    .otherwise(col("zona_del_hecho_cod"))
)

In [0]:
#filtrar en escenario_del_hecho_cod el valor "sin informacion"
df_escenario_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("escenario_del_hecho_cod") == "sin informacion")
display(df_escenario_sininfo)


Los valores "sin informacion" en las variables "escenario_del_hecho_cod", "actividad_durante_hecho_cod', 'sexo_del_agresor_cod' y 'presunto_agresor_cod', se definiran en función del valor típico en los grupos de la variable circunstancia_del_hecho_detallada.

In [0]:
#Filtrar variable "circuntancia_del_hecho_detallada" por Violencia contra niños, niñas y adolescentes
df_circunstancia_violencia_niños_adole= df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("circunstancia_del_hecho_detallada") == "Violencia contra niños, niñas y adolescentes")

In [0]:
df_circunstancia_violencia_niños_adole.count()

In [0]:
# Gráfica de barras escenario_del_hecho_cod 

#Agrupar por categoría, contar casos y ordenar
df_escenario_violencia_niños_adole = (
    df_circunstancia_violencia_niños_adole
    .groupBy("escenario_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("escenario_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_escenario_violencia_niños_adole.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["escenario_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos por escenario del hecho en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras actividad_durante_hecho_cod

#Agrupar por categoría, contar casos y ordenar
df_actividad_violencia_niños_adole = (
    df_circunstancia_violencia_niños_adole
    .groupBy("actividad_durante_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("actividad_durante_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_actividad_violencia_niños_adole.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["actividad_durante_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos por actividad durante el hecho en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")
plt.yticks(range(0, 22, 1))

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras sexo_del_agresor_cod

#Agrupar por categoría, contar casos y ordenar
df_sexo_violencia_niños_adole = (
    df_circunstancia_violencia_niños_adole
    .groupBy("sexo_del_agresor_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("sexo_del_agresor_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_sexo_violencia_niños_adole.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["sexo_del_agresor_cod"], pdf["cantidad"])

plt.title("Número de casos por sexo del agresor en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras presunto_agresor_cod

#Agrupar por categoría, contar casos y ordenar
df_agresor_violencia_niños_adole = (
    df_circunstancia_violencia_niños_adole
    .groupBy("presunto_agresor_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("presunto_agresor_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_agresor_violencia_niños_adole.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["presunto_agresor_cod"], pdf["cantidad"])

plt.title("Número de casos por presunto agresor en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Filtrar variable "circuntancia_del_hecho_detallada" por violencia de pareja
df_circunstancia_violencia_pareja= df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("circunstancia_del_hecho_detallada") == "Violencia de pareja")

In [0]:
df_circunstancia_violencia_pareja.count()

In [0]:
# Gráfica de barras escenario_del_hecho_cod 

#Agrupar por categoría, contar casos y ordenar
df_escenario_violencia_pareja = (
    df_circunstancia_violencia_pareja
    .groupBy("escenario_del_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("escenario_del_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_escenario_violencia_pareja.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["escenario_del_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos por escenario del hecho en violencia de pareja")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras actividad_durante_hecho_cod

#Agrupar por categoría, contar casos y ordenar
df_actividad_violencia_de_pareja = (
    df_circunstancia_violencia_pareja
    .groupBy("actividad_durante_hecho_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("actividad_durante_hecho_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_actividad_violencia_de_pareja.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["actividad_durante_hecho_cod"], pdf["cantidad"])

plt.title("Número de casos por actividad durante el hecho en violencia de pareja")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")
plt.yticks(range(0, 10, 1))

plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras sexo_del_agresor_cod

#Agrupar por categoría, contar casos y ordenar
df_sexo_violencia_de_pareja = (
    df_circunstancia_violencia_pareja
    .groupBy("sexo_del_agresor_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("sexo_del_agresor_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_sexo_violencia_de_pareja .toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["sexo_del_agresor_cod"], pdf["cantidad"])

plt.title("Número de casos por sexo del agresor en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Gráfica de barras presunto_agresor_cod

#Agrupar por categoría, contar casos y ordenar
df_agresor_violencia_de_pareja = (
    df_circunstancia_violencia_pareja
    .groupBy("presunto_agresor_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("presunto_agresor_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_agresor_violencia_de_pareja.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["presunto_agresor_cod"], pdf["cantidad"])

plt.title("Número de casos por presunto agresor en violencia contra niños y adolescentes")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Cambiar el valor sin información de escenario_del_hecho_cod por Vivienda

df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "escenario_del_hecho_cod",
    when(col("escenario_del_hecho_cod") == "sin informacion", "Vivienda")     
    .otherwise(col("escenario_del_hecho_cod"))
)

In [0]:
#filtrar en actividad_durante_hecho_cod el valor "sin informacion"
df_actividad_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("actividad_durante_hecho_cod") == "sin informacion")
display(df_actividad_sininfo)

In [0]:
#Cambiar el valor sin información de actividad_durante_hecho_cod por Actividades de desplazamiento de un lugar a otro

df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "actividad_durante_hecho_cod",
    when(col("actividad_durante_hecho_cod") == "sin informacion", "Actividades de desplazamiento de un lugar a otro")     
    .otherwise(col("actividad_durante_hecho_cod"))
)

In [0]:
#filtrar en sexo_del_agresor_cod el valor "sin informacion"
df_sexoagresor_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("sexo_del_agresor_cod") == "sin informacion")
display(df_sexoagresor_sininfo)

In [0]:
#Cambiar el valor sin informacion de sexo_del_agresor_cod por Hombre

df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "sexo_del_agresor_cod",
    when(col("sexo_del_agresor_cod") == "sin informacion", "Hombre")     
    .otherwise(col("sexo_del_agresor_cod"))
)

In [0]:
#filtrar en presunto_agresor_cod el valor "sin informacion"
df_agresor_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("presunto_agresor_cod") == "sin informacion")
display(df_agresor_sininfo)

In [0]:
#Cambiar el valor sin informacion de presunto_agresor_cod por Padre

df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "presunto_agresor_cod",
    when(col("presunto_agresor_cod") == "sin informacion", "Padre")     
    .otherwise(col("presunto_agresor_cod"))
)

In [0]:
#filtrar en factor_desencadenante_agresion_cod "sin informacion"
df_fdesencadenante_sininfo = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("factor_desencadenante_agresion_cod") == "sin informacion")
display(df_fdesencadenante_sininfo)

Los datos faltantes en factor_desencadenante_agresion_cod se van a completar en función a los casos predominantes segun el contexto_del_hecho.

In [0]:
#Filtrar variable "contexto_del_hecho" por Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar
df_contexto_violencia_ninos= df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("contexto_del_hecho") == "Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar")

In [0]:
df_contexto_violencia_ninos.count()

In [0]:
# Gráfica de barras contexto_del_hecho por Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar

#Agrupar por categoría, contar casos y ordenar
df_fdescencadenante_violencia_ninos= (
    df_contexto_violencia_ninos
    .groupBy("factor_desencadenante_agresion_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("factor_desencadenante_agresion_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_fdescencadenante_violencia_ninos.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["factor_desencadenante_agresion_cod"], pdf["cantidad"])

plt.title("Número de casos por factor desencadenante de la agresión en Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Filtrar variable "contexto_del_hecho" por Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar
df_contexto_violencia_pareja= df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col("contexto_del_hecho") == "Lesiones no Fatales por Violencia de Pareja")

In [0]:
df_contexto_violencia_pareja.count()

In [0]:
# Gráfica de barras contexto_del_hecho por Lesiones no Fatales por Violencia de Pareja

#Agrupar por categoría, contar casos y ordenar
df_fdescencadenante_violencia_pareja= (
    df_contexto_violencia_pareja
    .groupBy("factor_desencadenante_agresion_cod")
    .agg(count("*").alias("cantidad"))
    .orderBy(col("factor_desencadenante_agresion_cod"))
    .orderBy(col("cantidad").desc())
)

# Convertir a pandas
pdf = df_fdescencadenante_violencia_pareja.toPandas()

plt.figure(figsize=(10,5))

plt.bar(pdf["factor_desencadenante_agresion_cod"], pdf["cantidad"])

plt.title("Número de casos por factor desencadenante de la agresión en Lesiones no Fatales por Violencia de Pareja")
plt.xlabel("Departamento")
plt.ylabel("Cantidad")


plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
#Cambiar el valor sin informacion en "factor_desencadenante_agresion_cod"
df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "factor_desencadenante_agresion_cod",
    when(col("id") == "3", "Intolerancia, machismo")
    .when(col("id") == "23", "Intolerancia, machismo")
    .when(col("id") == "29", "Intolerancia, machismo")
    .when(col("id") == "53", "Intolerancia, machismo")
    .when(col("id") == "92", "Intolerancia, machismo")
    .when(col("id") == "108", "Celos, desconfianza, infidelidad")
    .when(col("id") == "109", "Intolerancia, machismo")
    .when(col("id") == "121", "Intolerancia, machismo")
    .when(col("id") == "134", "Intolerancia, machismo")
    .when(col("id") == "139", "Intolerancia, machismo")
    .when(col("id") == "144", "Intolerancia, machismo")
    .otherwise(col("factor_desencadenante_agresion_cod"))
)

In [0]:
#Cambiar el valor sin informacion en "dias_de_incapacidad_medicolegal_cod"

df_victimas_VIF_VP_con_muerte_homicidio_v1 = df_victimas_VIF_VP_con_muerte_homicidio_v1.withColumn(
    "dias_de_incapacidad_medicolegal_cod",
    when(col("dias_de_incapacidad_medicolegal_cod") == "sin informacion", "1 a 30 días")    
    .otherwise(col("dias_de_incapacidad_medicolegal_cod"))
)

In [0]:
#Verificar que no hayan valores "sin informacion" 

columnas_cod = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"

]

for columna in columnas_cod:
    cantidad = df_victimas_VIF_VP_con_muerte_homicidio_v1.filter(col(columna) == "sin informacion").count()

    if cantidad == 0:
        print(f"✅ La columna '{columna}' no tiene registros con 'sin informacion'.")
    else:
        print(f"❌ La columna '{columna}' tiene {cantidad} registros con 'sin informacion'.")

In [0]:
df_victimas_VIF_VP_con_muerte_homicidio_v1.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.victimas_VIF_VP_con_muerte_homicidio_v2"
    )